# GlyphMatics Nemotron Submission — v9 Deterministic Rebuild Transport

**Target:** competition-ready Kaggle notebook for the NVIDIA Nemotron Model Reasoning Challenge.

**Core change:** adds **Deterministic Subspace Rebuild** after PairFold, so discarded tail information is projected back into the same rank-32 survivor contract instead of tuning caps again.

The prior stable path did this:

```text
full fused delta -> SVD -> keep top 32 -> discard tail -> uniform gain restore -> RowGuard
```

This v8 path does this:

```text
full fused delta
-> SVD
-> keep top 32
-> fingerprint discarded tail directions by fused/layer block behavior
-> match tail directions to the closest surviving rank pair
-> redistribute restoration gain toward those matched pairs
-> optionally blend a tiny amount of matched tail orientation into the survivor pair
-> balanced A/B factorization
-> RowGuard clipping
-> validated submission.zip
```

Default submission contract:

| Variable | Default |
|---|---:|
| `FORCED_FUSED_RANK` | `32` |
| `SVD_ENERGY_GAIN_CAP` | `1.19` |
| `PAIRFOLD_ENABLED` | `1` |
| `PAIRFOLD_PAIR_WIDTH` | `2` |
| `PAIRFOLD_TAIL_MAX` | `96` |
| `PAIRFOLD_MIN_SIM` | `0.66` |
| `PAIRFOLD_MAX_GAIN_DELTA` | `0.022` |
| `PAIRFOLD_VECTOR_BLEND` | `0.025` |
| `ROW_NORM_GUARD_ENABLED` | `1` |
| `ROW_NORM_GAIN_CAP` | `1.08` |
| `DUAL_PAIR_ENABLED` | `0` |
| `GAIN_DISTRIBUTION` | `pairfold_residual_matched_balanced_rowguard` |

## Required Kaggle inputs

Attach these Kaggle inputs before running:

1. Competition input: NVIDIA Nemotron Model Reasoning Challenge
2. Base model: `nemotron-3-nano-30b-a3b-bf16`
3. Adapter input: `huikang/nemotron-adapter`
4. Local wheelhouse containing `tinker`, `tinker-cookbook`, and `chz` wheels

Internet is not required when the wheelhouse input is attached.


## v9 addition

This edition stops repeating gain-cap probes. It adds deterministic rebuild of lost residual information using a rank-safe subspace core projection: `core = Q_left.T @ Delta @ Q_right`. The adapter rank remains 32.


In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import importlib.util
import hashlib
import zipfile
import time
from collections import Counter

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

print("Python:", sys.version)
print("Working dir:", Path.cwd())

if not KAGGLE_INPUT.exists():
    raise RuntimeError("This notebook is intended to run inside Kaggle. Missing /kaggle/input.")

KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)

print("\n[Inputs]")
for p in sorted(KAGGLE_INPUT.iterdir()):
    print(" -", p)


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Working dir: /kaggle/working

[Inputs]
 - /kaggle/input/competitions
 - /kaggle/input/datasets
 - /kaggle/input/models


## 1. Install / load Tinker from local wheels

This scans Kaggle inputs for wheel folders and installs offline with `--no-index`.


In [2]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util
import importlib.metadata as md

def list_wheel_dirs():
    rows = []
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path("/tmp")]
    for root in roots:
        if not root.exists():
            continue
        for d in [root] + [p for p in root.rglob("*") if p.is_dir()]:
            wheels = sorted(d.glob("*.whl"))
            if wheels:
                rows.append((d, [w.name for w in wheels]))
    return rows

def score_tinker_dir(names):
    low = " ".join(n.lower() for n in names)
    score = 0
    required_hits = 0
    for token in ["tinker_cookbook", "tinker-cookbook", "tinker", "chz"]:
        if token in low:
            score += 10
            required_hits += 1
    score += min(len(names), 20)
    return score, required_hits

def find_tinker_wheelhouse():
    candidates = []
    for d, names in list_wheel_dirs():
        score, hits = score_tinker_dir(names)
        if hits:
            candidates.append((score, hits, len(names), d, names))
    candidates.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
    return candidates[0] if candidates else None

print("[Tinker] scanning wheel folders...")
wheel_rows = list_wheel_dirs()
for d, names in wheel_rows:
    interesting = [n for n in names if ("tinker" in n.lower() or "chz" in n.lower())]
    if interesting:
        print("\n[wheel-dir]", d)
        for name in interesting:
            print(" -", name)

if importlib.util.find_spec("tinker_cookbook") is None:
    explicit = os.environ.get("WHEEL_DIR", "").strip()
    candidate = None

    if explicit and Path(explicit).exists():
        candidate = (9999, 999, 0, Path(explicit), [p.name for p in Path(explicit).glob("*.whl")])
    else:
        candidate = find_tinker_wheelhouse()

    if candidate is None:
        raise FileNotFoundError(
            "Could not find local tinker wheelhouse. Attach a Kaggle input containing "
            "tinker-cookbook/tinker/chz wheels, or set WHEEL_DIR to that folder."
        )

    _, _, _, wheel_dir, names = candidate
    print("[Tinker] selected wheel_dir:", wheel_dir)

    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-index",
        f"--find-links={wheel_dir}",
        "tinker-cookbook",
        "tinker",
        "chz",
    ]
    print("[Tinker] pip:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("[Tinker] tinker_cookbook already installed; skipping wheel install")

import tinker_cookbook
from tinker_cookbook import weights

print("[Tinker] ready:", tinker_cookbook.__file__)
for pkg in ["tinker-cookbook", "tinker", "chz"]:
    try:
        print(f"[Tinker] {pkg} version:", md.version(pkg))
    except Exception as exc:
        print(f"[Tinker] {pkg} version unavailable:", exc)

if not hasattr(weights, "build_lora_adapter"):
    raise RuntimeError("tinker_cookbook.weights.build_lora_adapter not found")


[Tinker] scanning wheel folders...

[wheel-dir] /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
 - chz-0.4.0-py3-none-any.whl
 - tinker-0.18.1-py3-none-any.whl
 - tinker_cookbook-0.3.0-py3-none-any.whl
[Tinker] selected wheel_dir: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
[Tinker] pip: /usr/bin/python3 -m pip install --no-index --find-links=/kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse tinker-cookbook tinker chz
Looking in links: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker_cookbook-0.3.0-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker-0.18.1-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/chz-0.4.0-py3-none-any.whl
[Tinker] ready: /usr/local/lib/python3.12/dist-packages/tinker_cookbook/__init__.py
[Tinker] tinker-cookbook version: 0.3.0
[Tinker] tinker versi

## 2. Detect base model and adapter paths

The path finder prefers the known Kaggle model layout, then falls back to recursive detection.


In [3]:
from pathlib import Path
import json
import os

def find_first_existing(paths):
    for p in paths:
        q = Path(p)
        if q.exists():
            return q
    return None

ADAPTER_PATH_CANDIDATES = [
    "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/nemotron-adapter/transformers/default/20",
    "/kaggle/input/nemotron-adapter",
]

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16",
]

ADAPTER_PATH = find_first_existing(ADAPTER_PATH_CANDIDATES)
BASE_MODEL_PATH = find_first_existing(BASE_MODEL_CANDIDATES)

if ADAPTER_PATH is None:
    hits = []
    for cfg in Path("/kaggle/input").rglob("adapter_config.json"):
        folder = cfg.parent
        if (folder / "adapter_model.safetensors").exists():
            s = str(folder).lower()
            score = 0
            for token in ["huikang", "nemotron", "adapter"]:
                if token in s:
                    score += 10
            score -= len(str(folder)) / 1000
            hits.append((score, folder))
    if hits:
        hits.sort(reverse=True, key=lambda x: x[0])
        ADAPTER_PATH = hits[0][1]

if BASE_MODEL_PATH is None:
    hits = []
    for cfg in Path("/kaggle/input").rglob("config.json"):
        folder = cfg.parent
        s = str(folder).lower()
        if "adapter" in s:
            continue
        if "nemotron" in s and ("30b" in s or "nano" in s or "a3b" in s):
            shard_hits = list(folder.glob("*.safetensors")) + list(folder.glob("*.bin"))
            index_hits = list(folder.glob("*.index.json"))
            score = 100 + len(shard_hits) + len(index_hits) - len(str(folder)) / 1000
            hits.append((score, folder))
    if hits:
        hits.sort(reverse=True, key=lambda x: x[0])
        BASE_MODEL_PATH = hits[0][1]

if ADAPTER_PATH is None:
    raise FileNotFoundError("Nemotron adapter input not found. Attach huikang/nemotron-adapter.")

if BASE_MODEL_PATH is None:
    raise FileNotFoundError("Local Nemotron base model not found. Attach nemotron-3-nano-30b-a3b-bf16.")

ADAPTER_PATH = Path(ADAPTER_PATH)
BASE_MODEL_PATH = Path(BASE_MODEL_PATH)

print("[Paths] ADAPTER_PATH:", ADAPTER_PATH)
print("[Paths] BASE_MODEL_PATH:", BASE_MODEL_PATH)

if not (ADAPTER_PATH / "adapter_config.json").exists():
    raise FileNotFoundError(f"adapter_config.json missing under {ADAPTER_PATH}")
if not (ADAPTER_PATH / "adapter_model.safetensors").exists():
    raise FileNotFoundError(f"adapter_model.safetensors missing under {ADAPTER_PATH}")
if not (BASE_MODEL_PATH / "config.json").exists():
    raise FileNotFoundError(f"config.json missing under {BASE_MODEL_PATH}")

try:
    cfg = json.loads((BASE_MODEL_PATH / "config.json").read_text(encoding="utf-8"))
    print("[Base config] model_type:", cfg.get("model_type"))
    print("[Base config] architectures:", cfg.get("architectures"))
    print("[Base config] hidden_size:", cfg.get("hidden_size"))
    print("[Base config] num_hidden_layers:", cfg.get("num_hidden_layers"))
except Exception as exc:
    print("[Base config] non-fatal config read warning:", exc)


[Paths] ADAPTER_PATH: /kaggle/input/models/huikang/nemotron-adapter/transformers/default/20
[Paths] BASE_MODEL_PATH: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
[Base config] model_type: nemotron_h
[Base config] architectures: ['NemotronHForCausalLM']
[Base config] hidden_size: 2688
[Base config] num_hidden_layers: 52


## 3. Apply GlyphMatics v8 PairFold fused-projection transport patch

This patch intercepts Tinker fused projection merging and compresses oversized merged LoRA pairs to the required fused rank.

### What is fully defined here

| Area | Implementation |
|---|---|
| Fused merge | Builds one fused LoRA pair from separate projection components |
| Dense delta | Computes `Delta = B @ A` in fp32 |
| Rank compression | Uses SVD and keeps `FORCED_FUSED_RANK` directions |
| Residual matching | Builds block-energy signatures for kept and discarded directions |
| Pair compression | Groups kept ranks into pairs and matches tail residuals to closest pair |
| Gain transport | Redistributes safe restoration gain toward matched pairs |
| Optional tail orientation | Tiny capped vector blend into matched survivor directions |
| Balanced factorization | Splits scale symmetrically across LoRA `A` and `B` |
| RowGuard | Clips row-level overshoot after reconstruction |
| Ledger | Writes a transport ledger into the output adapter folder |

The defaults are intentionally conservative. They are designed to change *where* restoration is placed, not to brute-force more global energy.


In [4]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import os
import hashlib
import math

import torch
import tinker_cookbook.weights._adapter as A

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# -----------------------------------------------------------------------------
# Competition knobs
# -----------------------------------------------------------------------------
FORCED_FUSED_RANK = int(os.environ.get("FORCED_FUSED_RANK", "32"))
SVD_ENERGY_GAIN_CAP = float(os.environ.get("SVD_ENERGY_GAIN_CAP", "1.19"))
GAIN_DISTRIBUTION = "pairfold_residual_matched_deterministic_rebuild_rowguard"

DETERMINISTIC_REBUILD_ENABLED = os.environ.get("DETERMINISTIC_REBUILD_ENABLED", "1").strip().lower() not in {"0", "false", "no", "off"}
DETERMINISTIC_REBUILD_ACCEPT_EPS = float(os.environ.get("DETERMINISTIC_REBUILD_ACCEPT_EPS", "0.0000"))

ROW_NORM_GUARD_ENABLED = os.environ.get("ROW_NORM_GUARD_ENABLED", "1").strip().lower() not in {"0", "false", "no", "off"}
ROW_NORM_GAIN_CAP = float(os.environ.get("ROW_NORM_GAIN_CAP", "1.08"))

PAIRFOLD_ENABLED = os.environ.get("PAIRFOLD_ENABLED", "1").strip().lower() not in {"0", "false", "no", "off"}
PAIRFOLD_PAIR_WIDTH = int(os.environ.get("PAIRFOLD_PAIR_WIDTH", "2"))
PAIRFOLD_TAIL_MAX = int(os.environ.get("PAIRFOLD_TAIL_MAX", "96"))
PAIRFOLD_MIN_SIM = float(os.environ.get("PAIRFOLD_MIN_SIM", "0.66"))
PAIRFOLD_MAX_GAIN_DELTA = float(os.environ.get("PAIRFOLD_MAX_GAIN_DELTA", "0.022"))
PAIRFOLD_VECTOR_BLEND = float(os.environ.get("PAIRFOLD_VECTOR_BLEND", "0.025"))
PAIRFOLD_DECAY = float(os.environ.get("PAIRFOLD_DECAY", "0.985"))
PAIRFOLD_COL_BINS = int(os.environ.get("PAIRFOLD_COL_BINS", "8"))

DUAL_PAIR_ENABLED = os.environ.get("DUAL_PAIR_ENABLED", "0").strip().lower() not in {"0", "false", "no", "off"}
DUAL_PAIR_SPLIT_RAW = os.environ.get("DUAL_PAIR_SPLIT", "0.62,0.58")

if FORCED_FUSED_RANK <= 0:
    raise ValueError("FORCED_FUSED_RANK must be positive")
if not (1.0 <= SVD_ENERGY_GAIN_CAP <= 1.25):
    raise ValueError("SVD_ENERGY_GAIN_CAP must stay inside [1.0, 1.25]")
if not (1.0 <= ROW_NORM_GAIN_CAP <= 1.25):
    raise ValueError("ROW_NORM_GAIN_CAP must stay inside [1.0, 1.25]")
if PAIRFOLD_PAIR_WIDTH <= 0:
    raise ValueError("PAIRFOLD_PAIR_WIDTH must be positive")
if PAIRFOLD_TAIL_MAX < 0:
    raise ValueError("PAIRFOLD_TAIL_MAX must be non-negative")
if not (0.0 <= PAIRFOLD_MIN_SIM <= 1.0):
    raise ValueError("PAIRFOLD_MIN_SIM must be inside [0, 1]")
if not (0.0 <= PAIRFOLD_MAX_GAIN_DELTA <= 0.10):
    raise ValueError("PAIRFOLD_MAX_GAIN_DELTA must be inside [0, 0.10]")
if not (0.0 <= PAIRFOLD_VECTOR_BLEND <= 0.20):
    raise ValueError("PAIRFOLD_VECTOR_BLEND must be inside [0, 0.20]")
if not (0.50 <= PAIRFOLD_DECAY <= 1.0):
    raise ValueError("PAIRFOLD_DECAY must be inside [0.50, 1.0]")
if not (1 <= PAIRFOLD_COL_BINS <= 64):
    raise ValueError("PAIRFOLD_COL_BINS must be inside [1, 64]")
if not (-0.05 <= DETERMINISTIC_REBUILD_ACCEPT_EPS <= 0.05):
    raise ValueError("DETERMINISTIC_REBUILD_ACCEPT_EPS must be inside [-0.05, 0.05]")

def _parse_pair_split(raw: str):
    vals = []
    for piece in raw.replace(";", ",").split(","):
        piece = piece.strip()
        if piece:
            vals.append(float(piece))
    if len(vals) != 2:
        raise ValueError("DUAL_PAIR_SPLIT must contain exactly two numbers, e.g. '0.62,0.58'")
    if not all(v > 0 for v in vals):
        raise ValueError("DUAL_PAIR_SPLIT values must be positive")
    return vals

DUAL_PAIR_SPLIT = _parse_pair_split(DUAL_PAIR_SPLIT_RAW)

# -----------------------------------------------------------------------------
# Ledger
# -----------------------------------------------------------------------------
class GlyphmaticTransportLedger:
    def __init__(self):
        self.events = []
        self.counts = Counter()

    def emit(self, *, src, dst, op, alpha, beta=None, gamma=None):
        beta = beta or {}
        gamma = gamma or {}
        basis = json.dumps(
            {"alpha": alpha, "src": str(src), "dst": str(dst), "op": str(op), "beta": beta, "gamma": gamma},
            sort_keys=True,
            default=str,
        )
        event = {
            "alpha": str(alpha),
            "source": str(src),
            "dest": str(dst),
            "op": str(op),
            "beta": beta,
            "gamma": gamma,
            "verification_hash": hashlib.sha256(basis.encode("utf-8")).hexdigest()[:16],
        }
        self.events.append(event)
        self.counts[(event["alpha"], event["op"])] += 1

    def markdown(self) -> str:
        lines = [
            "# GlyphMatics Transport Ledger",
            "",
            "Generated during tinker-cookbook adapter conversion.",
            "",
            "## Submission configuration",
            "",
            f"- `FORCED_FUSED_RANK`: `{FORCED_FUSED_RANK}`",
            f"- `SVD_ENERGY_GAIN_CAP`: `{SVD_ENERGY_GAIN_CAP}`",
            f"- `GAIN_DISTRIBUTION`: `{GAIN_DISTRIBUTION}`",
            f"- `DETERMINISTIC_REBUILD_ENABLED`: `{DETERMINISTIC_REBUILD_ENABLED}`",
            f"- `DETERMINISTIC_REBUILD_ACCEPT_EPS`: `{DETERMINISTIC_REBUILD_ACCEPT_EPS}`",
            f"- `PAIRFOLD_ENABLED`: `{PAIRFOLD_ENABLED}`",
            f"- `PAIRFOLD_PAIR_WIDTH`: `{PAIRFOLD_PAIR_WIDTH}`",
            f"- `PAIRFOLD_TAIL_MAX`: `{PAIRFOLD_TAIL_MAX}`",
            f"- `PAIRFOLD_MIN_SIM`: `{PAIRFOLD_MIN_SIM}`",
            f"- `PAIRFOLD_MAX_GAIN_DELTA`: `{PAIRFOLD_MAX_GAIN_DELTA}`",
            f"- `PAIRFOLD_VECTOR_BLEND`: `{PAIRFOLD_VECTOR_BLEND}`",
            f"- `PAIRFOLD_DECAY`: `{PAIRFOLD_DECAY}`",
            f"- `PAIRFOLD_COL_BINS`: `{PAIRFOLD_COL_BINS}`",
            f"- `ROW_NORM_GUARD_ENABLED`: `{ROW_NORM_GUARD_ENABLED}`",
            f"- `ROW_NORM_GAIN_CAP`: `{ROW_NORM_GAIN_CAP}`",
            f"- `DUAL_PAIR_ENABLED`: `{DUAL_PAIR_ENABLED}`",
            f"- `DUAL_PAIR_SPLIT`: `{DUAL_PAIR_SPLIT}`",
            "",
            "## Event summary",
            "",
            "| alpha | op | count |",
            "|---|---|---:|",
        ]
        for (alpha, op), count in sorted(self.counts.items()):
            lines.append(f"| `{alpha}` | `{op}` | {count} |")

        lines += [
            "",
            "## First 80 events",
            "",
            "| # | alpha | op | source | destination | gamma | hash |",
            "|---:|---|---|---|---|---|---|",
        ]
        for i, event in enumerate(self.events[:80], 1):
            gamma = json.dumps(event["gamma"], sort_keys=True, default=str)
            lines.append(
                f"| {i} | `{event['alpha']}` | `{event['op']}` | "
                f"`{event['source']}` | `{event['dest']}` | `{gamma}` | `{event['verification_hash']}` |"
            )
        return "\n".join(lines) + "\n"

    def print_summary(self):
        print("[GlyphMatics ledger] events:", len(self.events))
        for (alpha, op), count in sorted(self.counts.items()):
            print(f"[GlyphMatics ledger] {alpha}:{op}={count}")

GLYPH_LEDGER = GlyphmaticTransportLedger()

# -----------------------------------------------------------------------------
# Numeric helpers
# -----------------------------------------------------------------------------
def _safe_unit(x: torch.Tensor, dim=None, eps: float = 1e-12):
    if dim is None:
        return x / torch.linalg.vector_norm(x).clamp_min(eps)
    return x / torch.linalg.vector_norm(x, dim=dim, keepdim=True).clamp_min(eps)

def _make_even_blocks(length: int, count: int, prefix: str):
    count = max(1, min(int(count), int(length)))
    blocks = []
    for i in range(count):
        start = int(round(i * length / count))
        end = int(round((i + 1) * length / count))
        if end > start:
            blocks.append((start, end, f"{prefix}{i}"))
    return blocks

def _make_row_blocks(row_count: int, component_slices=None):
    blocks = []
    if component_slices:
        for row_start, row_end, _rank, name in component_slices:
            row_start = max(0, min(int(row_start), int(row_count)))
            row_end = max(row_start, min(int(row_end), int(row_count)))
            if row_end > row_start:
                blocks.append((row_start, row_end, str(name)))
    covered = sum(e - s for s, e, _ in blocks)
    if not blocks or covered < row_count:
        # Fallback also covers non-component rows in unusual fused layouts.
        blocks = _make_even_blocks(row_count, min(8, row_count), "rowbin")
    return blocks

def _block_energy(vec: torch.Tensor, blocks):
    vals = []
    vec = vec.float()
    sq = vec * vec
    for start, end, _name in blocks:
        vals.append(sq[start:end].sum())
    out = torch.stack(vals) if vals else torch.ones(1, device=vec.device, dtype=torch.float32)
    return _safe_unit(out.float())

def _direction_signature(u_col: torch.Tensor, vh_row: torch.Tensor, row_blocks, col_bins: int):
    """
    Local behavior signature for residual matching.

    Raw SVD vectors are orthogonal globally, so direct cosine is not useful.
    This signature compares where a direction spends energy by fused output rows
    and input-column bins. Tail directions are folded into survivor pairs with
    similar local behavior.
    """
    col_blocks = _make_even_blocks(int(vh_row.numel()), max(1, min(col_bins, int(vh_row.numel()))), "colbin")
    row_sig = _block_energy(u_col, row_blocks)
    col_sig = _block_energy(vh_row, col_blocks)
    u_abs = u_col.float().abs()
    v_abs = vh_row.float().abs()
    moments = torch.tensor(
        [
            float(u_abs.max().detach().cpu()),
            float(v_abs.max().detach().cpu()),
            float(u_abs.mean().detach().cpu()),
            float(v_abs.mean().detach().cpu()),
        ],
        device=u_col.device,
        dtype=torch.float32,
    )
    return _safe_unit(torch.cat([row_sig, col_sig, _safe_unit(moments)]).float())

def _rank_pair_groups(rank: int, pair_width: int):
    groups = []
    start = 0
    while start < rank:
        end = min(rank, start + pair_width)
        groups.append(list(range(start, end)))
        start = end
    return groups

def _dual_lane_gain_vector(*, singular_values: torch.Tensor, raw_gain: torch.Tensor, rank: int):
    """
    Stable two-lane option retained for A/B tests.

    By default DUAL_PAIR_ENABLED=0 because PairFold is now the primary transport.
    """
    base_gain = torch.clamp(raw_gain, min=1.0, max=SVD_ENERGY_GAIN_CAP)
    gain_vec = torch.full_like(singular_values, fill_value=float(base_gain))

    if not DUAL_PAIR_ENABLED or rank < 2:
        return gain_vec, {
            "mode": "single_lane_before_pairfold",
            "raw_energy_gain": float(raw_gain.detach().cpu()),
            "global_energy_gain": float(base_gain.detach().cpu()),
            "lane_ranks": [int(rank)],
            "lane_caps": [float(SVD_ENERGY_GAIN_CAP)],
            "lane_gains": [float(base_gain.detach().cpu())],
        }

    first = rank // 2
    second = rank - first
    pair_mean = sum(DUAL_PAIR_SPLIT) / 2.0
    cap_excess = max(SVD_ENERGY_GAIN_CAP - 1.0, 0.0)

    lane_caps = [
        min(SVD_ENERGY_GAIN_CAP, 1.0 + cap_excess * (DUAL_PAIR_SPLIT[0] / pair_mean)),
        min(SVD_ENERGY_GAIN_CAP, 1.0 + cap_excess * (DUAL_PAIR_SPLIT[1] / pair_mean)),
    ]

    lane_gain_0 = torch.clamp(raw_gain, min=1.0, max=lane_caps[0])
    lane_gain_1 = torch.clamp(raw_gain, min=1.0, max=lane_caps[1])

    gain_vec[:first] = lane_gain_0
    gain_vec[first:first + second] = lane_gain_1

    return gain_vec, {
        "mode": "dual_pair_before_pairfold",
        "raw_energy_gain": float(raw_gain.detach().cpu()),
        "global_energy_gain_cap": float(SVD_ENERGY_GAIN_CAP),
        "dual_pair_split": [float(x) for x in DUAL_PAIR_SPLIT],
        "lane_ranks": [int(first), int(second)],
        "lane_caps": [float(x) for x in lane_caps],
        "lane_gains": [float(lane_gain_0.detach().cpu()), float(lane_gain_1.detach().cpu())],
    }

def _pairfold_transport(U: torch.Tensor, S: torch.Tensor, Vh: torch.Tensor, rank: int, gain_vec: torch.Tensor, row_blocks):
    """
    Residual-matched rank-pair compression.

    The discarded SVD tail is not directly retained as extra rank. Instead:
    1. Each kept direction gets a local behavior signature.
    2. Adjacent kept directions are grouped into rank pairs.
    3. Tail directions are matched to the closest surviving pair by signature.
    4. Restoration gain is redistributed toward pairs that absorbed similar tail.
    5. A tiny capped orientation blend can nudge the survivor pair toward the tail.

    Global gain is energy-renormalized back near the incoming target so this
    changes transport shape more than total volume.
    """
    device = U.device
    rank = int(rank)
    tail_available = max(0, int(S.numel()) - rank)
    if (not PAIRFOLD_ENABLED) or rank <= 0 or tail_available <= 0 or PAIRFOLD_TAIL_MAX <= 0:
        return U[:, :rank].contiguous(), Vh[:rank, :].contiguous(), gain_vec.contiguous(), {
            "pairfold_enabled": bool(PAIRFOLD_ENABLED),
            "pairfold_mode": "disabled_or_no_tail",
            "pairfold_assignments": 0,
            "pairfold_tail_used": 0,
        }

    U_k = U[:, :rank].clone()
    Vh_k = Vh[:rank, :].clone()
    S_k = S[:rank]
    gain_in = gain_vec.clone()

    head_sigs = torch.stack([
        _direction_signature(U[:, i], Vh[i, :], row_blocks, PAIRFOLD_COL_BINS)
        for i in range(rank)
    ])

    groups = _rank_pair_groups(rank, PAIRFOLD_PAIR_WIDTH)
    group_sigs = []
    for group in groups:
        idx = torch.tensor(group, device=device, dtype=torch.long)
        weights = S_k[idx].float().clamp_min(1e-12)
        weights = weights / weights.sum().clamp_min(1e-12)
        sig = (head_sigs[idx] * weights.unsqueeze(1)).sum(dim=0)
        group_sigs.append(_safe_unit(sig))
    group_sigs = torch.stack(group_sigs)

    pair_priority = torch.zeros(len(groups), device=device, dtype=torch.float32)
    head_priority = torch.zeros(rank, device=device, dtype=torch.float32)
    u_acc = torch.zeros_like(U_k.float())
    v_acc = torch.zeros_like(Vh_k.float())

    tail_count = min(tail_available, int(PAIRFOLD_TAIL_MAX))
    assignments = 0
    sim_sum = 0.0
    max_sim = 0.0

    for local_j in range(tail_count):
        j = rank + local_j
        tail_sig = _direction_signature(U[:, j], Vh[j, :], row_blocks, PAIRFOLD_COL_BINS)
        sims = group_sigs @ tail_sig
        best_group = int(torch.argmax(sims).detach().cpu())
        best_sim = float(sims[best_group].detach().cpu())
        if best_sim < PAIRFOLD_MIN_SIM:
            continue

        decay = float(PAIRFOLD_DECAY ** local_j)
        # Priority is relative, not raw energy injection. It tells where the
        # already-safe restoration should be placed.
        rel_tail = float((S[j] / S_k.mean().clamp_min(1e-12)).detach().cpu())
        priority = max(0.0, best_sim * decay * rel_tail)
        if priority <= 0:
            continue

        assignments += 1
        sim_sum += best_sim
        max_sim = max(max_sim, best_sim)
        pair_priority[best_group] += priority

        group = groups[best_group]
        idx = torch.tensor(group, device=device, dtype=torch.long)
        local_sims = (head_sigs[idx] @ tail_sig).clamp_min(0.0)
        local_weights = local_sims + S_k[idx].float() / S_k[idx].float().sum().clamp_min(1e-12)
        local_weights = local_weights / local_weights.sum().clamp_min(1e-12)

        for pos, head_idx in enumerate(group):
            head_priority[head_idx] += priority * float(local_weights[pos].detach().cpu())
            if PAIRFOLD_VECTOR_BLEND > 0:
                # Tiny orientation fold. Capped per tail and scaled by similarity.
                blend = min(
                    PAIRFOLD_VECTOR_BLEND,
                    PAIRFOLD_VECTOR_BLEND * best_sim * rel_tail,
                ) * float(local_weights[pos].detach().cpu())
                u_acc[:, head_idx] += float(blend) * U[:, j].float()
                v_acc[head_idx, :] += float(blend) * Vh[j, :].float()

    if assignments == 0 or float(head_priority.sum().detach().cpu()) <= 0.0:
        return U_k.contiguous(), Vh_k.contiguous(), gain_in.contiguous(), {
            "pairfold_enabled": True,
            "pairfold_mode": "no_similarity_match",
            "pairfold_assignments": 0,
            "pairfold_tail_used": int(tail_count),
            "pairfold_min_sim": float(PAIRFOLD_MIN_SIM),
        }

    # Rank-pair gain redistribution.
    # Center around 1.0, cap the delta, then energy-renormalize back to the
    # pre-PairFold target. This avoids simply making the adapter louder.
    priority = head_priority / head_priority.mean().clamp_min(1e-12)
    delta = (priority - 1.0).clamp(-1.0, 1.0) * float(PAIRFOLD_MAX_GAIN_DELTA)
    gain_out = gain_in * (1.0 + delta)
    gain_out = torch.clamp(gain_out, min=1.0, max=float(SVD_ENERGY_GAIN_CAP))

    target_energy = torch.sqrt(torch.sum((S_k * gain_in) ** 2)).clamp_min(1e-12)
    current_energy = torch.sqrt(torch.sum((S_k * gain_out) ** 2)).clamp_min(1e-12)
    gain_out = gain_out * (target_energy / current_energy)
    gain_out = torch.clamp(gain_out, min=1.0, max=float(SVD_ENERGY_GAIN_CAP))

    if PAIRFOLD_VECTOR_BLEND > 0:
        U_k = _safe_unit(U_k.float() + u_acc, dim=0).to(U.dtype)
        Vh_k = _safe_unit(Vh_k.float() + v_acc, dim=1).to(Vh.dtype)

    return U_k.contiguous(), Vh_k.contiguous(), gain_out.contiguous(), {
        "pairfold_enabled": True,
        "pairfold_mode": "residual_matched_rank_pairs",
        "pairfold_pair_width": int(PAIRFOLD_PAIR_WIDTH),
        "pairfold_tail_used": int(tail_count),
        "pairfold_assignments": int(assignments),
        "pairfold_assignment_rate": float(assignments / max(1, tail_count)),
        "pairfold_avg_match_sim": float(sim_sum / max(1, assignments)),
        "pairfold_max_match_sim": float(max_sim),
        "pairfold_min_sim": float(PAIRFOLD_MIN_SIM),
        "pairfold_max_gain_delta": float(PAIRFOLD_MAX_GAIN_DELTA),
        "pairfold_vector_blend": float(PAIRFOLD_VECTOR_BLEND),
        "pairfold_gain_mean": float(gain_out.mean().detach().cpu()),
        "pairfold_gain_min": float(gain_out.min().detach().cpu()),
        "pairfold_gain_max": float(gain_out.max().detach().cpu()),
    }

def _apply_row_norm_guard(B_new: torch.Tensor, A_new: torch.Tensor, delta: torch.Tensor):
    """Clip local row-level overshoot after compression."""
    if not ROW_NORM_GUARD_ENABLED:
        return B_new.contiguous(), {
            "row_norm_guard_enabled": False,
            "row_norm_gain_cap": float(ROW_NORM_GAIN_CAP),
            "row_clip_fraction": 0.0,
        }

    with torch.no_grad():
        recon = B_new.float() @ A_new.float()
        orig_row = torch.linalg.vector_norm(delta.float(), ord=2, dim=1).clamp_min(1e-12)
        new_row = torch.linalg.vector_norm(recon, ord=2, dim=1).clamp_min(1e-12)
        ratio = new_row / orig_row
        max_allowed = orig_row * float(ROW_NORM_GAIN_CAP)
        row_scale = torch.minimum(torch.ones_like(new_row), max_allowed / new_row)
        clipped = row_scale < 0.999
        B_guarded = (B_new.float() * row_scale.unsqueeze(1)).to(B_new.dtype).contiguous()

    return B_guarded, {
        "row_norm_guard_enabled": True,
        "row_norm_gain_cap": float(ROW_NORM_GAIN_CAP),
        "row_clip_fraction": float(clipped.float().mean().detach().cpu()),
        "row_clip_count": int(clipped.sum().detach().cpu()),
        "row_ratio_mean_before_guard": float(ratio.mean().detach().cpu()),
        "row_ratio_max_before_guard": float(ratio.max().detach().cpu()),
    }


def _factorize_balanced_from_basis(U_basis: torch.Tensor, core: torch.Tensor, V_basis: torch.Tensor):
    """
    Factor a projected rank-k core into LoRA B/A factors.

    U_basis: [out_dim, k] orthonormal columns
    core:    [k, k] projected dense delta
    V_basis: [in_dim, k] orthonormal columns

    Produces:
      B_new: [out_dim, k]
      A_new: [k, in_dim]
    """
    P, Sc, Qh = torch.linalg.svd(core.float(), full_matrices=False)
    Sc = Sc.float().clamp_min(0.0)
    sroot = torch.sqrt(Sc.clamp_min(1e-12))
    B_new = (U_basis.float() @ P.float()) * sroot.unsqueeze(0)
    A_new = sroot.unsqueeze(1) * (Qh.float() @ V_basis.float().T)
    return B_new.contiguous(), A_new.contiguous(), Sc.contiguous()

def _rebuild_objective(B_new: torch.Tensor, A_new: torch.Tensor, delta: torch.Tensor):
    """
    Deterministic accept metric.

    Primary term is global residual. Secondary term penalizes row overshoot.
    This keeps the rebuild from winning merely by becoming louder.
    """
    recon = B_new.float() @ A_new.float()
    delta_norm = torch.linalg.vector_norm(delta.float()).clamp_min(1e-12)
    residual = torch.linalg.vector_norm((delta.float() - recon).float()) / delta_norm

    orig_row = torch.linalg.vector_norm(delta.float(), ord=2, dim=1).clamp_min(1e-12)
    new_row = torch.linalg.vector_norm(recon.float(), ord=2, dim=1).clamp_min(1e-12)
    row_ratio = new_row / orig_row
    overshoot = torch.clamp(row_ratio - float(ROW_NORM_GAIN_CAP), min=0.0).mean()

    return residual + 0.05 * overshoot, residual, overshoot

def _deterministic_subspace_rebuild(
    U_k: torch.Tensor,
    Vh_k: torch.Tensor,
    S_k: torch.Tensor,
    gain_vec: torch.Tensor,
    delta: torch.Tensor,
    baseline_B: torch.Tensor,
    baseline_A: torch.Tensor,
    baseline_row_guard_stats: dict,
):
    """
    Deterministic lost-information rebuild inside the same rank contract.

    PairFold changes the survivor directions, but the old diagonal factorization
    still only uses one singular lane at a time. This rebuild treats the survivor
    left/right directions as a rank-k subspace and computes the best k x k core
    projection of the original dense delta inside that subspace:

        core = Q_left.T @ Delta @ Q_right

    That core contains cross-lane residual information that the diagonal path
    discards. We then SVD-factor the core back into standard LoRA B/A tensors.
    No extra rank is created, no randomness is used, and the output still obeys
    the same `(out_dim, rank)` / `(rank, in_dim)` evaluator contract.
    """
    if not DETERMINISTIC_REBUILD_ENABLED:
        return baseline_B.contiguous(), baseline_A.contiguous(), {
            "deterministic_rebuild_enabled": False,
            "deterministic_rebuild_mode": "disabled",
        }, baseline_row_guard_stats

    if U_k.numel() == 0 or Vh_k.numel() == 0:
        return baseline_B.contiguous(), baseline_A.contiguous(), {
            "deterministic_rebuild_enabled": True,
            "deterministic_rebuild_mode": "empty_basis_fallback",
        }, baseline_row_guard_stats

    with torch.no_grad():
        base_obj, base_resid, base_overshoot = _rebuild_objective(baseline_B, baseline_A, delta)

        # Build orthonormal survivor bases. QR gives deterministic bases for a
        # fixed input matrix and avoids relying on non-orthogonal blended lanes.
        Q_left, _ = torch.linalg.qr(U_k.float(), mode="reduced")
        Q_right, _ = torch.linalg.qr(Vh_k.float().T, mode="reduced")

        core = Q_left.T @ delta.float() @ Q_right
        B_core, A_core, Sc_core = _factorize_balanced_from_basis(Q_left, core, Q_right)

        # Keep total restored energy aligned to the existing cap/gain target.
        target_energy = torch.sqrt(torch.sum((S_k.float() * gain_vec.float()) ** 2)).clamp_min(1e-12)
        core_energy = torch.sqrt(torch.sum(Sc_core.float() ** 2)).clamp_min(1e-12)
        energy_scale = (target_energy / core_energy).clamp(0.25, 4.0)
        if torch.isfinite(energy_scale):
            scale_root = torch.sqrt(energy_scale)
            B_core = B_core * scale_root
            A_core = A_core * scale_root

        B_core, rebuild_row_guard_stats = _apply_row_norm_guard(B_core, A_core, delta)
        cand_obj, cand_resid, cand_overshoot = _rebuild_objective(B_core, A_core, delta)

        accept = bool(cand_obj <= base_obj * (1.0 + float(DETERMINISTIC_REBUILD_ACCEPT_EPS)))

    stats = {
        "deterministic_rebuild_enabled": True,
        "deterministic_rebuild_mode": "accepted_subspace_core_projection" if accept else "rejected_kept_pairfold_diagonal",
        "deterministic_rebuild_accepted": bool(accept),
        "deterministic_rebuild_base_objective": float(base_obj.detach().cpu()),
        "deterministic_rebuild_candidate_objective": float(cand_obj.detach().cpu()),
        "deterministic_rebuild_base_residual": float(base_resid.detach().cpu()),
        "deterministic_rebuild_candidate_residual": float(cand_resid.detach().cpu()),
        "deterministic_rebuild_base_overshoot": float(base_overshoot.detach().cpu()),
        "deterministic_rebuild_candidate_overshoot": float(cand_overshoot.detach().cpu()),
        "deterministic_rebuild_energy_scale": float(energy_scale.detach().cpu()),
        "deterministic_rebuild_core_energy": float(core_energy.detach().cpu()),
        "deterministic_rebuild_target_energy": float(target_energy.detach().cpu()),
    }

    if accept:
        return B_core.contiguous(), A_core.contiguous(), stats, rebuild_row_guard_stats
    return baseline_B.contiguous(), baseline_A.contiguous(), stats, baseline_row_guard_stats


def _compress_lora_pair_to_rank(B: torch.Tensor, A_mat: torch.Tensor, rank: int, component_slices=None):
    """
    Compress Delta = B @ A to rank-k with PairFold residual matching.

    The output shape is always `(out_dim, rank)` and `(rank, in_dim)` unless
    the underlying matrix is smaller than the requested rank.
    """
    if B.ndim != 2 or A_mat.ndim != 2:
        raise ValueError(f"Expected 2D LoRA matrices, got B={tuple(B.shape)}, A={tuple(A_mat.shape)}")
    if B.shape[1] != A_mat.shape[0]:
        raise ValueError(f"LoRA inner rank mismatch: B={tuple(B.shape)}, A={tuple(A_mat.shape)}")

    delta = B.float() @ A_mat.float()
    if not torch.isfinite(delta).all():
        raise FloatingPointError("Dense LoRA delta contains non-finite values before compression")

    max_rank = min(delta.shape)
    rank = min(int(rank), int(max_rank))
    if rank <= 0:
        raise ValueError(f"Invalid compression rank {rank} for delta shape {tuple(delta.shape)}")

    U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
    total_mass = S.sum().clamp_min(1e-12)
    full_energy = torch.sqrt(torch.sum(S ** 2)).clamp_min(1e-12)

    U_k = U[:, :rank]
    S_k = S[:rank]
    Vh_k = Vh[:rank, :]

    kept_energy = torch.sqrt(torch.sum(S_k ** 2)).clamp_min(1e-12)
    raw_gain = full_energy / kept_energy
    gain_vec, gain_stats = _dual_lane_gain_vector(
        singular_values=S_k,
        raw_gain=raw_gain,
        rank=rank,
    )

    row_blocks = _make_row_blocks(int(delta.shape[0]), component_slices=component_slices)
    U_k, Vh_k, gain_vec, pairfold_stats = _pairfold_transport(
        U=U,
        S=S,
        Vh=Vh,
        rank=rank,
        gain_vec=gain_vec,
        row_blocks=row_blocks,
    )

    # Baseline balanced transport: split scale symmetrically across both LoRA factors.
    sroot_balanced = torch.sqrt(S_k * gain_vec)
    B_baseline = U_k * sroot_balanced.unsqueeze(0)
    A_baseline = sroot_balanced.unsqueeze(1) * Vh_k
    B_baseline, row_guard_stats = _apply_row_norm_guard(B_baseline, A_baseline, delta)

    # Deterministic rebuild: recover residual cross-lane information inside the
    # same rank-32 evaluator contract, then accept only if the local objective
    # does not regress.
    B_new, A_new, rebuild_stats, row_guard_stats = _deterministic_subspace_rebuild(
        U_k=U_k,
        Vh_k=Vh_k,
        S_k=S_k,
        gain_vec=gain_vec,
        delta=delta,
        baseline_B=B_baseline,
        baseline_A=A_baseline,
        baseline_row_guard_stats=row_guard_stats,
    )

    restored_energy = torch.sqrt(torch.sum((S_k * gain_vec) ** 2)).clamp_min(1e-12)
    recon = B_new.float() @ A_new.float()
    residual_energy = torch.linalg.vector_norm((delta - recon).float()).clamp_min(1e-12)

    if not torch.isfinite(B_new).all() or not torch.isfinite(A_new).all():
        raise FloatingPointError("Compressed LoRA factors contain non-finite values")

    stats = {
        "rank_in": int(B.shape[1]),
        "rank_out": int(rank),
        "delta_shape": [int(delta.shape[0]), int(delta.shape[1])],
        "singular_mass_kept": float((S_k.sum() / total_mass).detach().cpu()),
        "energy_kept_ratio": float((kept_energy / full_energy).detach().cpu()),
        "energy_after_gain_ratio": float((restored_energy / full_energy).detach().cpu()),
        "reconstruction_residual_ratio": float((residual_energy / torch.linalg.vector_norm(delta).clamp_min(1e-12)).detach().cpu()),
        "gain_distribution": GAIN_DISTRIBUTION,
        "row_block_count": int(len(row_blocks)),
        "row_blocks": [{"name": name, "rows": [int(start), int(end)]} for start, end, name in row_blocks],
        **gain_stats,
        **pairfold_stats,
        **rebuild_stats,
        **row_guard_stats,
    }

    return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous(), stats

# -----------------------------------------------------------------------------
# Tinker patch: fused projection merge
# -----------------------------------------------------------------------------
def patched_merge_fused_projections(
    fused_model_key: str,
    adapter_layer_prefix: str,
    components,
    model_state_shapes,
    peft_weights,
    target_modules,
    profile,
) -> int:
    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix(".weight").rsplit(".", 1)[-1]

    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break
    if component_order is None:
        raise RuntimeError(f"No fused projection component order found for {fused_target_name!r}")

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts = []
    comp_slices = []
    merged_rank = 0
    row_offset = 0

    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(f"Missing component {comp_name!r} for fused target {fused_model_key!r}")

        lora_A, lora_B = comp_by_name[comp_name]
        if lora_A.ndim != 2 or lora_B.ndim != 2:
            raise ValueError(f"Expected 2D LoRA tensors for {comp_name}: A={tuple(lora_A.shape)}, B={tuple(lora_B.shape)}")
        if lora_A.shape[0] != lora_B.shape[1]:
            raise ValueError(f"Component rank mismatch for {comp_name}: A={tuple(lora_A.shape)}, B={tuple(lora_B.shape)}")

        r = int(lora_A.shape[0])
        out_dim = int(lora_B.shape[0])

        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + out_dim, r, comp_name))
        row_offset += out_dim
        merged_rank += r

    merged_lora_A = torch.cat(lora_A_parts, dim=0)
    merged_lora_B = torch.zeros(
        fused_out_dim,
        merged_rank,
        dtype=merged_lora_A.dtype,
        device=merged_lora_A.device,
    )

    rank_offset = 0
    for row_start, row_end, r, comp_name in comp_slices:
        _, lora_B = comp_by_name[comp_name]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    final_rank = int(merged_rank)
    compression_stats = {
        "rank_in": int(merged_rank),
        "rank_out": int(merged_rank),
        "preservation": "exact_no_compression",
        "component_row_total": int(row_offset),
        "fused_out_dim": int(fused_out_dim),
    }

    if merged_rank > FORCED_FUSED_RANK:
        merged_lora_B, merged_lora_A, svd_stats = _compress_lora_pair_to_rank(
            merged_lora_B,
            merged_lora_A,
            FORCED_FUSED_RANK,
            component_slices=comp_slices,
        )
        final_rank = int(merged_lora_A.shape[0])
        compression_stats = {
            **svd_stats,
            "preservation": "rank32_pairfold_deterministic_rebuild_rowguard",
            "gain_cap": float(SVD_ENERGY_GAIN_CAP),
        }

    peft_target_key = f"{adapter_layer_prefix}.{fused_target_name}.weight"

    GLYPH_LEDGER.emit(
        src=f"{adapter_layer_prefix}.{{{','.join(component_order)}}}",
        dst=peft_target_key,
        op="fused_projection_pairfold_transport",
        alpha="mamba_or_fused_projection",
        beta={
            "fused_model_key": fused_model_key,
            "fused_out_dim": int(fused_out_dim),
            "component_count": len(component_order),
            "component_order": list(component_order),
            "component_slices": [
                {"name": name, "rows": [int(a), int(b)], "rank": int(r)}
                for a, b, r, name in comp_slices
            ],
        },
        gamma=compression_stats,
    )

    A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
    return final_rank

if not hasattr(A, "_merge_fused_projections"):
    raise RuntimeError("tinker_cookbook.weights._adapter._merge_fused_projections not found")

A._merge_fused_projections = patched_merge_fused_projections

print("[GlyphMatics] patched:", A._merge_fused_projections.__name__)
print("[GlyphMatics] FORCED_FUSED_RANK:", FORCED_FUSED_RANK)
print("[GlyphMatics] SVD_ENERGY_GAIN_CAP:", SVD_ENERGY_GAIN_CAP)
print("[GlyphMatics] GAIN_DISTRIBUTION:", GAIN_DISTRIBUTION)
print("[GlyphMatics] DETERMINISTIC_REBUILD_ENABLED:", DETERMINISTIC_REBUILD_ENABLED)
print("[GlyphMatics] DETERMINISTIC_REBUILD_ACCEPT_EPS:", DETERMINISTIC_REBUILD_ACCEPT_EPS)
print("[GlyphMatics] PAIRFOLD_ENABLED:", PAIRFOLD_ENABLED)
print("[GlyphMatics] PAIRFOLD_PAIR_WIDTH:", PAIRFOLD_PAIR_WIDTH)
print("[GlyphMatics] PAIRFOLD_TAIL_MAX:", PAIRFOLD_TAIL_MAX)
print("[GlyphMatics] PAIRFOLD_MIN_SIM:", PAIRFOLD_MIN_SIM)
print("[GlyphMatics] PAIRFOLD_MAX_GAIN_DELTA:", PAIRFOLD_MAX_GAIN_DELTA)
print("[GlyphMatics] PAIRFOLD_VECTOR_BLEND:", PAIRFOLD_VECTOR_BLEND)
print("[GlyphMatics] ROW_NORM_GUARD_ENABLED:", ROW_NORM_GUARD_ENABLED)
print("[GlyphMatics] ROW_NORM_GAIN_CAP:", ROW_NORM_GAIN_CAP)
print("[GlyphMatics] DUAL_PAIR_ENABLED:", DUAL_PAIR_ENABLED)
print("[GlyphMatics] DUAL_PAIR_SPLIT:", DUAL_PAIR_SPLIT)


[GlyphMatics] patched: patched_merge_fused_projections
[GlyphMatics] FORCED_FUSED_RANK: 32
[GlyphMatics] SVD_ENERGY_GAIN_CAP: 1.19
[GlyphMatics] GAIN_DISTRIBUTION: pairfold_residual_matched_deterministic_rebuild_rowguard
[GlyphMatics] DETERMINISTIC_REBUILD_ENABLED: True
[GlyphMatics] DETERMINISTIC_REBUILD_ACCEPT_EPS: 0.0
[GlyphMatics] PAIRFOLD_ENABLED: True
[GlyphMatics] PAIRFOLD_PAIR_WIDTH: 2
[GlyphMatics] PAIRFOLD_TAIL_MAX: 96
[GlyphMatics] PAIRFOLD_MIN_SIM: 0.66
[GlyphMatics] PAIRFOLD_MAX_GAIN_DELTA: 0.022
[GlyphMatics] PAIRFOLD_VECTOR_BLEND: 0.025
[GlyphMatics] ROW_NORM_GUARD_ENABLED: True
[GlyphMatics] ROW_NORM_GAIN_CAP: 1.08
[GlyphMatics] DUAL_PAIR_ENABLED: False
[GlyphMatics] DUAL_PAIR_SPLIT: [0.62, 0.58]


## 4. Local compression self-tests

These tests run before building the adapter. They validate the new PairFold path using synthetic LoRA tensors, without requiring the full model weights to be loaded.


In [5]:
import torch

print("[SelfTest] deterministic PairFold compression tests")
_gen = torch.Generator(device="cpu").manual_seed(918)

B_test = torch.randn(48, 24, generator=_gen, dtype=torch.float32)
A_test = torch.randn(24, 40, generator=_gen, dtype=torch.float32)
component_slices_test = [
    (0, 12, 8, "q_proj"),
    (12, 24, 8, "k_proj"),
    (24, 36, 4, "v_proj"),
    (36, 48, 4, "gate_proj"),
]

B_comp, A_comp, stats = _compress_lora_pair_to_rank(
    B_test,
    A_test,
    rank=8,
    component_slices=component_slices_test,
)

assert tuple(B_comp.shape) == (48, 8), tuple(B_comp.shape)
assert tuple(A_comp.shape) == (8, 40), tuple(A_comp.shape)
assert torch.isfinite(B_comp).all(), "B_comp contains non-finite values"
assert torch.isfinite(A_comp).all(), "A_comp contains non-finite values"

orig_delta = B_test @ A_test
new_delta = B_comp.float() @ A_comp.float()
assert torch.isfinite(new_delta).all(), "reconstructed delta contains non-finite values"

if ROW_NORM_GUARD_ENABLED:
    orig_row = torch.linalg.vector_norm(orig_delta, dim=1).clamp_min(1e-12)
    new_row = torch.linalg.vector_norm(new_delta, dim=1).clamp_min(1e-12)
    worst_ratio = float((new_row / orig_row).max().detach().cpu())
    assert worst_ratio <= ROW_NORM_GAIN_CAP + 2e-4, (worst_ratio, ROW_NORM_GAIN_CAP)

assert stats["rank_out"] == 8, stats
assert stats["pairfold_enabled"] == bool(PAIRFOLD_ENABLED), stats
assert 0.0 <= stats["energy_kept_ratio"] <= 1.000001, stats
assert stats["reconstruction_residual_ratio"] >= 0.0, stats

print("[SelfTest] passed")
print(json.dumps({
    "rank_in": stats["rank_in"],
    "rank_out": stats["rank_out"],
    "energy_kept_ratio": stats["energy_kept_ratio"],
    "energy_after_gain_ratio": stats["energy_after_gain_ratio"],
    "pairfold_mode": stats.get("pairfold_mode"),
    "pairfold_assignments": stats.get("pairfold_assignments"),
    "row_clip_fraction": stats.get("row_clip_fraction"),
    "reconstruction_residual_ratio": stats["reconstruction_residual_ratio"],
}, indent=2))

assert stats.get("deterministic_rebuild_enabled") is True, "Deterministic rebuild flag missing"
assert stats.get("deterministic_rebuild_mode") in {
    "accepted_subspace_core_projection",
    "rejected_kept_pairfold_diagonal",
    "empty_basis_fallback",
}, stats.get("deterministic_rebuild_mode")
print("[SelfTest] deterministic rebuild mode:", stats.get("deterministic_rebuild_mode"))
print("[SelfTest] deterministic rebuild accepted:", stats.get("deterministic_rebuild_accepted"))


[SelfTest] deterministic PairFold compression tests
[SelfTest] passed
{
  "rank_in": 24,
  "rank_out": 8,
  "energy_kept_ratio": 0.8670838475227356,
  "energy_after_gain_ratio": 1.0,
  "pairfold_mode": "residual_matched_rank_pairs",
  "pairfold_assignments": 32,
  "row_clip_fraction": 0.1458333283662796,
  "reconstruction_residual_ratio": 0.5145506262779236
}
[SelfTest] deterministic rebuild mode: accepted_subspace_core_projection
[SelfTest] deterministic rebuild accepted: True


## 5. Build adapter package

This clears stale output, builds the converted adapter, and creates required marker files.


In [6]:
from pathlib import Path
import shutil
import json
import time
from tinker_cookbook import weights

OUTPUT_DIR = Path("/kaggle/working/nemotron-adapter-ready-to-submit")
ZIP_PATH = Path("/kaggle/working/submission.zip")

if OUTPUT_DIR.exists():
    print("[Build] removing stale output:", OUTPUT_DIR)
    shutil.rmtree(OUTPUT_DIR)

print("[Build] adapter_path:", ADAPTER_PATH)
print("[Build] base_model_path:", BASE_MODEL_PATH)
print("[Build] output_dir:", OUTPUT_DIR)

t0 = time.time()
weights.build_lora_adapter(
    base_model=str(BASE_MODEL_PATH),
    adapter_path=str(ADAPTER_PATH),
    output_path=str(OUTPUT_DIR),
)
elapsed = time.time() - t0
print(f"[Build] build_lora_adapter completed in {elapsed/60:.2f} min")

required_core = [
    "adapter_config.json",
    "adapter_model.safetensors",
]

missing_core = [name for name in required_core if not (OUTPUT_DIR / name).exists()]
if missing_core:
    raise FileNotFoundError(f"Adapter build did not produce required files: {missing_core}")

# Required by some submit/eval paths.
(OUTPUT_DIR / "checkpoint_complete").write_text("ok\n", encoding="utf-8")

GLYPH_LEDGER.print_summary()
ledger_text = GLYPH_LEDGER.markdown()
(OUTPUT_DIR / "GLYPHMATICS_TRANSPORT_LEDGER.md").write_text(ledger_text, encoding="utf-8")

readme_text = [
    "# Nemotron Adapter Submission",
    "",
    "GlyphMatics v8 PairFold residual-matched row-guarded fused-projection transport.",
    "",
    "## Build knobs",
    "",
    f"- FORCED_FUSED_RANK: {FORCED_FUSED_RANK}",
    f"- SVD_ENERGY_GAIN_CAP: {SVD_ENERGY_GAIN_CAP}",
    f"- GAIN_DISTRIBUTION: {GAIN_DISTRIBUTION}",
    f"- PAIRFOLD_ENABLED: {PAIRFOLD_ENABLED}",
    f"- PAIRFOLD_PAIR_WIDTH: {PAIRFOLD_PAIR_WIDTH}",
    f"- PAIRFOLD_TAIL_MAX: {PAIRFOLD_TAIL_MAX}",
    f"- PAIRFOLD_MIN_SIM: {PAIRFOLD_MIN_SIM}",
    f"- PAIRFOLD_MAX_GAIN_DELTA: {PAIRFOLD_MAX_GAIN_DELTA}",
    f"- PAIRFOLD_VECTOR_BLEND: {PAIRFOLD_VECTOR_BLEND}",
    f"- ROW_NORM_GUARD_ENABLED: {ROW_NORM_GUARD_ENABLED}",
    f"- ROW_NORM_GAIN_CAP: {ROW_NORM_GAIN_CAP}",
    f"- DUAL_PAIR_ENABLED: {DUAL_PAIR_ENABLED}",
    f"- DUAL_PAIR_SPLIT: {DUAL_PAIR_SPLIT}",
    "",
]
readme_path = OUTPUT_DIR / "README.md"
if readme_path.exists():
    existing = readme_path.read_text(encoding="utf-8", errors="replace")
    readme_path.write_text(existing.rstrip() + "\n\n" + "\n".join(readme_text) + "\n", encoding="utf-8")
else:
    readme_path.write_text("\n".join(readme_text) + "\n", encoding="utf-8")

manifest = {
    "name": "glyphmatics_nemotron_v8_pairfold_competition_ready",
    "output_dir": str(OUTPUT_DIR),
    "elapsed_seconds": elapsed,
    "forced_fused_rank": FORCED_FUSED_RANK,
    "svd_energy_gain_cap": SVD_ENERGY_GAIN_CAP,
    "gain_distribution": GAIN_DISTRIBUTION,
    "pairfold_enabled": PAIRFOLD_ENABLED,
    "pairfold_pair_width": PAIRFOLD_PAIR_WIDTH,
    "pairfold_tail_max": PAIRFOLD_TAIL_MAX,
    "pairfold_min_sim": PAIRFOLD_MIN_SIM,
    "pairfold_max_gain_delta": PAIRFOLD_MAX_GAIN_DELTA,
    "pairfold_vector_blend": PAIRFOLD_VECTOR_BLEND,
    "row_norm_gain_cap": ROW_NORM_GAIN_CAP,
    "dual_pair_enabled": DUAL_PAIR_ENABLED,
    "dual_pair_split": DUAL_PAIR_SPLIT,
    "ledger_events": len(GLYPH_LEDGER.events),
}
(OUTPUT_DIR / "submission_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("[Build] output files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name, p.stat().st_size)


[Build] adapter_path: /kaggle/input/models/huikang/nemotron-adapter/transformers/default/20
[Build] base_model_path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
[Build] output_dir: /kaggle/working/nemotron-adapter-ready-to-submit


MoE expert LoRA serving for nemotron models is experimental in vLLM and not yet supported in SGLang. The adapter will be produced but may not work with all serving configurations.


[Build] build_lora_adapter completed in 3.09 min
[GlyphMatics ledger] events: 23
[GlyphMatics ledger] mamba_or_fused_projection:fused_projection_pairfold_transport=23
[Build] output files:
 - GLYPHMATICS_TRANSPORT_LEDGER.md 59342
 - README.md 539
 - adapter_config.json 618
 - adapter_model.safetensors 3554384888
 - checkpoint_complete 3
 - submission_manifest.json 626


## 6. Validate and create `submission.zip`

The zip is intentionally minimal: only the files most likely required by the evaluator are included.


In [7]:
import zipfile
import hashlib
from pathlib import Path
import json

OUTPUT_DIR = Path("/kaggle/working/nemotron-adapter-ready-to-submit")
ZIP_PATH = Path("/kaggle/working/submission.zip")

required = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "README.md",
    "checkpoint_complete",
]

missing = [name for name in required if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing required submission files: {missing}")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in required:
        p = OUTPUT_DIR / name
        zf.write(p, arcname=name)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    bad = zf.testzip()

print("[Zip] wrote:", ZIP_PATH)
print("[Zip] size:", ZIP_PATH.stat().st_size)
print("[Zip] sha256:", sha256_file(ZIP_PATH))
print("[Zip] contents:")
for name in names:
    print(" -", name)

if bad is not None:
    raise RuntimeError(f"Zip integrity check failed at member: {bad}")
if ZIP_PATH.stat().st_size <= 0:
    raise RuntimeError("submission.zip is empty")
if names != required:
    raise RuntimeError(f"Unexpected zip contents/order: {names}")

print("[Zip] validation passed")


[Zip] wrote: /kaggle/working/submission.zip
[Zip] size: 3270300980
[Zip] sha256: ee684b0a180b7df2f53f1e74716131d93fb83187d4ac5395e560913895ac5ee4
[Zip] contents:
 - adapter_config.json
 - adapter_model.safetensors
 - README.md
 - checkpoint_complete
[Zip] validation passed


## 7. Final listing

Submit `/kaggle/working/submission.zip`.


In [8]:
from pathlib import Path
import os

for p in [
    Path("/kaggle/working/submission.zip"),
    Path("/kaggle/working/nemotron-adapter-ready-to-submit"),
]:
    print("\n[Listing]", p)
    if p.is_file():
        print(p, p.stat().st_size)
    elif p.is_dir():
        for child in sorted(p.iterdir()):
            print(" -", child.name, child.stat().st_size)
    else:
        print("MISSING:", p)



[Listing] /kaggle/working/submission.zip
/kaggle/working/submission.zip 3270300980

[Listing] /kaggle/working/nemotron-adapter-ready-to-submit
 - GLYPHMATICS_TRANSPORT_LEDGER.md 59342
 - README.md 539
 - adapter_config.json 618
 - adapter_model.safetensors 3554384888
 - checkpoint_complete 3
 - submission_manifest.json 626
